# CausalMark — Evaluation Notebook

Evaluates Criteo uplift predictions, treatment-group summaries, confidence intervals, targeting strategies, and saved model metrics.

**Important:** predicted uplift and factual prediction accuracy do not, by themselves, establish causal impact.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/Users/naitikdonda/Desktop/CasualMark v3")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "evaluation"
METRICS_DIR = PROJECT_ROOT / "notebooks" / "artifacts" / "metrics"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CRITEO_PATH = DATA_DIR / "marketing_instructions" / "criteo_sample_50000.csv"
criteo_df = pd.read_csv(CRITEO_PATH)

print("Rows:", len(criteo_df))
print("Columns:", criteo_df.columns.tolist())
display(criteo_df.head())


## 1. Treatment-group descriptive summary

In [ ]:
group_summary = (
    criteo_df.groupby("treatment")
    .agg(
        customers=("treatment", "size"),
        conversion_rate=("conversion", "mean"),
        visit_rate=("visit", "mean"),
        exposure_rate=("exposure", "mean")
    )
    .reset_index()
)

display(group_summary)

group_summary.to_csv(
    OUTPUT_DIR / "criteo_treatment_group_summary.csv",
    index=False
)


## 2. Conversion confidence intervals

In [ ]:
ci_rows = []

for treatment_value, group in criteo_df.groupby("treatment"):
    n = len(group)
    p = group["conversion"].mean()
    se = np.sqrt(p * (1 - p) / n)
    ci_rows.append({
        "treatment": treatment_value,
        "customers": n,
        "conversion_rate": p,
        "ci_lower": max(0.0, p - norm.ppf(0.975) * se),
        "ci_upper": min(1.0, p + norm.ppf(0.975) * se)
    })

conversion_ci_df = pd.DataFrame(ci_rows)
display(conversion_ci_df)

conversion_ci_df.to_csv(
    OUTPUT_DIR / "criteo_conversion_confidence_intervals.csv",
    index=False
)


## 3. Load and review existing model metrics

In [ ]:
metrics_path = METRICS_DIR / "criteo_final_metrics.csv"

if metrics_path.exists():
    model_metrics_df = pd.read_csv(metrics_path)
    display(model_metrics_df)
else:
    model_metrics_df = pd.DataFrame()
    print("Model metrics file not found:", metrics_path)


## 4. Uplift targeting analysis

In [ ]:
predictions_path = METRICS_DIR / "criteo_test_uplift_predictions.csv"

if predictions_path.exists():
    predictions = pd.read_csv(predictions_path)
    uplift_columns = [
        col for col in predictions.columns
        if "uplift" in col.lower()
    ]

    if uplift_columns:
        uplift_col = uplift_columns[0]
        scores = pd.to_numeric(
            predictions[uplift_col], errors="coerce"
        ).dropna().sort_values(ascending=False).reset_index(drop=True)

        targeting_rows = []
        for fraction in [0.05, 0.10, 0.20, 0.30, 0.50, 1.00]:
            n_selected = max(1, int(np.ceil(len(scores) * fraction)))
            selected = scores.iloc[:n_selected]
            targeting_rows.append({
                "target_fraction": fraction,
                "customers_selected": n_selected,
                "mean_predicted_uplift": selected.mean(),
                "sum_predicted_uplift": selected.sum()
            })

        targeting_df = pd.DataFrame(targeting_rows)
        display(targeting_df)
        targeting_df.to_csv(
            OUTPUT_DIR / "criteo_uplift_strategy_comparison.csv",
            index=False
        )

        plt.figure(figsize=(8, 4))
        plt.plot(
            targeting_df["target_fraction"] * 100,
            targeting_df["mean_predicted_uplift"],
            marker="o"
        )
        plt.xlabel("Customers targeted (%)")
        plt.ylabel("Mean predicted uplift")
        plt.title("Predicted Uplift by Targeting Fraction")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("No uplift column found:", predictions.columns.tolist())
else:
    print("Prediction file not found:", predictions_path)


## 5. Review final evaluation status

In [ ]:
report_path = OUTPUT_DIR / "causalmark_final_evaluation_package.json"

if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as file:
        final_package = json.load(file)

    print(json.dumps(final_package, indent=2))
else:
    print("Final evaluation package not found:", report_path)

print("\\nCausal effect verification must remain false until supported by a valid causal evaluation.")
